# Thử nghiệm Đối chứng với Thư viện Scikit-Surprise

Notebook này sử dụng thư viện `scikit-surprise` để chạy các thuật toán tương tự trên cùng một tập dữ liệu MovieLens 100k.
Kết quả MAE và RMSE ở đây sẽ được dùng để đối chiếu trực tiếp với kết quả chạy từ thư mục `src/` (tự code từ đầu).

In [1]:
import os
import pandas as pd
from surprise import Dataset, Reader, KNNBaseline, SVD, accuracy
from surprise.model_selection import train_test_split

# Đọc dữ liệu
data_path = '../data/raw/u.data'
reader = Reader(line_format='user item rating timestamp', sep='\t')
data = Dataset.load_from_file(data_path, reader=reader)

# Chia tập Train/Test (80/20)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print("--- Đánh giá Thư viện: KNNBaseline (User-Based, Pearson) ---")
# Tương đương với: Custom User-Based CF (Biased Baseline) của chúng ta
algo_knn_user = KNNBaseline(k=40, sim_options={'name': 'pearson_baseline', 'user_based': True})
algo_knn_user.fit(trainset)
preds_knn_user = algo_knn_user.test(testset)
accuracy.rmse(preds_knn_user)
accuracy.mae(preds_knn_user)

print("\n--- Đánh giá Thư viện: KNNBaseline (Item-Based, Pearson) ---")
# Tương đương với: Custom Item-Based CF của chúng ta
algo_knn_item = KNNBaseline(k=40, sim_options={'name': 'pearson_baseline', 'user_based': False})
algo_knn_item.fit(trainset)
preds_knn_item = algo_knn_item.test(testset)
accuracy.rmse(preds_knn_item)
accuracy.mae(preds_knn_item)

print("\n--- Đánh giá Thư viện: SVD ---")
# Tương đương với: Custom SVD của chúng ta
algo_svd = SVD(n_factors=20, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo_svd.fit(trainset)
preds_svd = algo_svd.test(testset)
accuracy.rmse(preds_svd)
accuracy.mae(preds_svd)


--- Đánh giá Thư viện: KNNBaseline (User-Based, Pearson) ---
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
RMSE: 0.9197
MAE:  0.7190

--- Đánh giá Thư viện: KNNBaseline (Item-Based, Pearson) ---
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
RMSE: 0.9169
MAE:  0.7188

--- Đánh giá Thư viện: SVD ---
RMSE: 0.9350
MAE:  0.7370


np.float64(0.7369633476148726)

In [5]:
print("\n======================================================\n")
print("=== ĐỐI CHỨNG VỚI MÔ HÌNH TỰ XÂY DỰNG TỪ ĐẦU (SCRATCH) ===\n")
print("======================================================\n")
import sys
sys.path.append("../")
from src.data_loader import load_matrix
from src.evaluation import compute_mae, compute_rmse
import pickle

test_matrix = load_matrix("../data/processed/test_matrix.npy")

with open("../models/user_cf.pkl", "rb") as f:
    user_cf = pickle.load(f)
with open("../models/item_cf.pkl", "rb") as f:
    item_cf = pickle.load(f)
    
from src.recommender import MatrixFactorizationSVD
svd_model = MatrixFactorizationSVD()
svd_model.load_model("../models/svd_weights.pkl")

print("\n--- Đánh giá Custom User-Based CF (Biased Baseline) ---")
print(f"RMSE: {compute_rmse(test_matrix, user_cf):.4f}")
print(f"MAE:  {compute_mae(test_matrix, user_cf):.4f}")

print("\n--- Đánh giá Custom Item-Based CF ---")
print(f"RMSE: {compute_rmse(test_matrix, item_cf):.4f}")
print(f"MAE:  {compute_mae(test_matrix, item_cf):.4f}")

print("\n--- Đánh giá Custom SVD ---")
print(f"RMSE: {compute_rmse(test_matrix, svd_model):.4f}")
print(f"MAE:  {compute_mae(test_matrix, svd_model):.4f}")




=== ĐỐI CHỨNG VỚI MÔ HÌNH TỰ XÂY DỰNG TỪ ĐẦU (SCRATCH) ===



--- Đánh giá Custom User-Based CF (Biased Baseline) ---
RMSE: 0.9261
MAE:  0.7262

--- Đánh giá Custom Item-Based CF ---


AttributeError: 'ItemBasedCollaborativeFiltering' object has no attribute 'item_means'